# Golden Set Data-Gap Evaluation — Test Sample

목적:

1. `app.stream()`의 노드별 결과를 구조화된 trace로 수집한다.
2. Golden Claim별 데이터 가용성과 실제 retrieval을 비교한다.
3. **원천 데이터 부족 vs Agent 실패**를 구분한다.
4. 문항별 root cause를 출력한다.

> 이 노트북은 `app`이 없어도 synthetic trace로 실행된다. 실제 프로젝트에서는 `app`을 import한 뒤 `USE_SYNTHETIC=False`로 변경하면 된다.


In [ ]:
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional
import pandas as pd

pd.set_option("display.max_colwidth", 120)

## 1. 평가용 Golden Schema

실제 운영에서는 아래 dict들을 `evaluation/golden_eval.jsonl`로 분리하는 것을 권장한다.

샘플은 서로 다른 판정 유형을 포함한다.

- Q2: 산출가능
- Q11: 정의변경
- Q23: 부분산출
- Q24: 외부데이터필요
- Q31: ABSTAIN_INVALID_TAXONOMY
- Q35: ABSTAIN_DOMAIN_MISMATCH


In [ ]:

SAMPLE_GOLDEN = [
    {
        "question_id": "Q2",
        "question": "국고채권 02000-3106(21-5)의 발행일, 만기일, 잔존일수, 표면금리와 세후수익률을 알려줘. 상품번호와 데이터 갱신일을 근거로 붙여줘.",
        "golden_data_status": "산출가능",
        "expected_action": "ANSWER",
        "expected_routes": ["rdb"],
        "claims": [
            {"claim_id": "C1", "label": "상품번호", "source": "rdb", "availability": "available",
             "required_fields": ["pd_no"], "answer_tokens": ["KR103502GB66"]},
            {"claim_id": "C2", "label": "발행일", "source": "rdb", "availability": "available",
             "required_fields": ["isu_dt"], "answer_tokens": ["20210610"]},
            {"claim_id": "C3", "label": "만기일", "source": "rdb", "availability": "available",
             "required_fields": ["mat_dt"], "answer_tokens": ["20310610"]},
            {"claim_id": "C4", "label": "잔존일수", "source": "rdb", "availability": "available",
             "required_fields": ["remaining_days"], "answer_tokens": ["1754"]},
            {"claim_id": "C5", "label": "표면금리", "source": "rdb", "availability": "available",
             "required_fields": ["srfc_irt"], "answer_tokens": ["2"]},
            {"claim_id": "C6", "label": "세후수익률", "source": "rdb", "availability": "available",
             "required_fields": ["after_tax_yield"], "answer_tokens": ["3.9279"]},
            {"claim_id": "C7", "label": "기준일", "source": "rdb", "availability": "available",
             "required_fields": ["info_base_dt"], "answer_tokens": ["20260821"]},
        ]
    },
    {
        "question_id": "Q11",
        "question": "최신 갱신일 기준 매수가능수량이 0보다 큰 원화채권 중 신용등급이 AA- 이상인 종목을 알려줘.",
        "golden_data_status": "정의변경",
        "expected_action": "ANSWER_WITH_REDEFINED_RULE",
        "expected_routes": ["rdb"],
        "forbidden_fields": ["buyable_quantity"],
        "required_definition_tokens": ["만기", "미도래"],
        "claims": [
            {"claim_id": "C1", "label": "구매가능 재정의", "source": "rdb",
             "availability": "deprecated_definition", "required_fields": ["mat_dt"]},
            {"claim_id": "C2", "label": "신용등급", "source": "rdb",
             "availability": "available", "required_fields": ["crd_grd"]},
            {"claim_id": "C3", "label": "수익률", "source": "rdb",
             "availability": "available", "required_fields": ["applied_yield"]},
        ]
    },
    {
        "question_id": "Q23",
        "question": "최근 6개월 동안 우주항공 테마와 연결된 이력이 있는 ETF를 정리해줘.",
        "golden_data_status": "부분산출",
        "expected_action": "ANSWER_PARTIAL_WITH_GAP",
        "expected_routes": ["rdb", "graph", "vector"],
        "claims": [
            {"claim_id": "C1", "label": "현재 우주항공/방산 테마 ETF", "source": "rdb",
             "availability": "available", "required_fields": ["themes"]},
            {"claim_id": "C2", "label": "최근 6개월 테마 연결 이력", "source": "graph",
             "availability": "external_required", "required_relations": ["THEME_HISTORY"],
             "gap_tokens": ["6개월", "이력", "시점", "날짜", "외부"]},
            {"claim_id": "C3", "label": "현재 편입관계와 뉴스 언급 구분", "source": "vector",
             "availability": "external_required", "required_fields": ["holding_as_of", "event_date"],
             "gap_tokens": ["편입", "내역", "필요"]},
        ]
    },
    {
        "question_id": "Q24",
        "question": "에코프로의 자회사를 편입한 ETF를 찾고 그중 최신 AUM이 가장 큰 상품의 위험요인을 알려줘.",
        "golden_data_status": "외부데이터필요",
        "expected_action": "ANSWER_WITH_EXTERNAL_GAP",
        "expected_routes": ["graph", "rdb", "vector"],
        "claims": [
            {"claim_id": "C1", "label": "에코프로 자회사", "source": "graph",
             "availability": "external_required", "required_relations": ["SUBSIDIARY_OF"],
             "gap_tokens": ["자회사", "외부", "DART"]},
            {"claim_id": "C2", "label": "자회사 ETF 편입관계", "source": "graph",
             "availability": "external_required", "required_relations": ["HOLDS_SECURITY"],
             "gap_tokens": ["편입", "내역"]},
            {"claim_id": "C3", "label": "편입비중", "source": "graph",
             "availability": "external_required", "required_fields": ["holding_weight", "holding_as_of"],
             "gap_tokens": ["편입비중", "편입", "내역"]},
            {"claim_id": "C4", "label": "ETF AUM", "source": "rdb",
             "availability": "available", "required_fields": ["pd_net_tamt"]},
            {"claim_id": "C5", "label": "위험요인 문서", "source": "vector",
             "availability": "external_required", "required_fields": ["document_name", "evidence_text"],
             "gap_tokens": ["위험", "투자설명서", "문서"]},
        ]
    },
    {
        "question_id": "Q31",
        "question": "신용등급이 AAAA인 매수 가능 채권을 찾아 상품번호와 신용평가 근거를 제시해줘.",
        "golden_data_status": "ABSTAIN",
        "expected_action": "ABSTAIN_INVALID_TAXONOMY",
        "expected_routes": [],
        "claims": [
            {"claim_id": "C1", "label": "신용등급 AAAA", "source": "ontology",
             "availability": "invalid_taxonomy",
             "abstain_tokens": ["ABSTAIN_INVALID_TAXONOMY", "AAAA", "존재하지"]}
        ]
    },
    {
        "question_id": "Q35",
        "question": "VOO가 직접 발행한 회사채의 신용등급과 만기일을 알려줘.",
        "golden_data_status": "ABSTAIN",
        "expected_action": "ABSTAIN_DOMAIN_MISMATCH",
        "expected_routes": [],
        "claims": [
            {"claim_id": "C1", "label": "VOO의 회사채 발행", "source": "ontology",
             "availability": "domain_mismatch",
             "abstain_tokens": ["ABSTAIN_DOMAIN_MISMATCH", "ETF", "Bond"]}
        ]
    }
]

len(SAMPLE_GOLDEN)


## 2. Agent 실행 Trace 수집기

기존 `test_rdb_pipeline()`은 출력 중심이므로 평가 자동화를 위해 **dict를 반환하는 함수**로 바꾼다.


In [ ]:

def safe_json_loads(value):
    if isinstance(value, dict):
        return value
    if not isinstance(value, str):
        return {"raw": value}
    try:
        return json.loads(value)
    except Exception:
        return {"raw": value}


def collect_agent_trace(app, question: str, max_sql_retries: Optional[int] = None) -> Dict[str, Any]:
    q_id = f"Q-{uuid.uuid4().hex[:6].upper()}"
    inputs = {"question_id": q_id, "question": question}
    if max_sql_retries is not None:
        inputs["max_sql_retries"] = max_sql_retries

    trace = {
        "runtime_question_id": q_id,
        "question": question,
        "nodes_executed": [],
        "intent": None,
        "verified_intent": None,
        "plan": [],
        "route": {},
        "step_results": {},
        "merged_rows": [],
        "answer": {},
        "errors": [],
        "trace_messages": []
    }

    for output in app.stream(inputs):
        for node_name, update in output.items():
            trace["nodes_executed"].append(node_name)

            for msg in update.get("trace", []) or []:
                trace["trace_messages"].append(str(msg))

            if node_name == "analyze_intent":
                trace["intent"] = update.get("intent")

            elif node_name == "verify_intent":
                trace["verified_intent"] = update.get("intent")

            elif node_name == "plan_query":
                trace["plan"] = update.get("plan", []) or []
                trace["route"] = update.get("route", {}) or {}

            elif node_name in {"rdb_search", "graph_search", "vector_search"}:
                step_results = update.get("step_results", {}) or {}
                trace["step_results"].setdefault(node_name, {}).update(step_results)

                for step_id, result in step_results.items():
                    if isinstance(result, dict) and result.get("error"):
                        trace["errors"].append({
                            "node": node_name,
                            "step_id": step_id,
                            "error": result.get("error")
                        })

            elif node_name == "merge_results":
                trace["merged_rows"] = update.get("merged_rows", []) or []

            elif node_name == "generate_answer":
                trace["answer"] = safe_json_loads(update.get("answer", {}))

    return trace


## 3. Evidence 추출

검색 결과의 row key, SQL/SPARQL, merged row key를 모아 Claim이 요구하는 field/relation이 실제로 검색되었는지 검사한다.

> 운영 버전에서는 `step_results`의 schema를 통일해 `evidence_items`를 명시적으로 반환하는 편이 더 좋다.


In [ ]:

def flatten_dict_keys(obj: Any, prefix: str = "") -> set:
    keys = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            keys.add(str(k).lower())
            keys.add(key.lower())
            keys |= flatten_dict_keys(v, key)
    elif isinstance(obj, list):
        for item in obj:
            keys |= flatten_dict_keys(item, prefix)
    return keys


def trace_text(trace: Dict[str, Any]) -> str:
    chunks = [
        json.dumps(trace.get("plan", []), ensure_ascii=False, default=str),
        json.dumps(trace.get("route", {}), ensure_ascii=False, default=str),
        json.dumps(trace.get("step_results", {}), ensure_ascii=False, default=str),
        json.dumps(trace.get("merged_rows", []), ensure_ascii=False, default=str),
        json.dumps(trace.get("answer", {}), ensure_ascii=False, default=str),
        " ".join(trace.get("trace_messages", []))
    ]
    return "\n".join(chunks).lower()


def answer_text(trace: Dict[str, Any]) -> str:
    ans = trace.get("answer", {})
    if isinstance(ans, dict):
        return " ".join(str(v) for v in ans.values()).lower()
    return str(ans).lower()


def executed_sources(trace: Dict[str, Any]) -> set:
    nodes = set(trace.get("nodes_executed", []))
    result = set()
    if "rdb_search" in nodes:
        result.add("rdb")
    if "graph_search" in nodes:
        result.add("graph")
    if "vector_search" in nodes:
        result.add("vector")
    # ontology validity may be handled before DB route
    if trace.get("verified_intent") is not None:
        result.add("ontology")
    return result


def evidence_found_for_claim(claim: Dict[str, Any], trace: Dict[str, Any]) -> bool:
    keys = flatten_dict_keys({
        "step_results": trace.get("step_results", {}),
        "merged_rows": trace.get("merged_rows", [])
    })
    text = trace_text(trace)

    required_fields = [x.lower() for x in claim.get("required_fields", [])]
    required_relations = [x.lower() for x in claim.get("required_relations", [])]

    field_ok = True if not required_fields else any(
        f in keys or f in text for f in required_fields
    )
    relation_ok = True if not required_relations else any(
        r in text for r in required_relations
    )
    return field_ok and relation_ok


## 4. Claim 진단 규칙

중요한 순서:

1. 질의가 taxonomy/domain/cutoff/definition상 유효한지
2. Golden상 데이터가 존재해야 하는지
3. 필요한 route가 실행됐는지
4. retrieval evidence가 있는지
5. answer가 evidence를 사용했는지


In [ ]:

ABSTAIN_MAP = {
    "invalid_taxonomy": "ABSTAIN_INVALID_TAXONOMY",
    "domain_mismatch": "ABSTAIN_DOMAIN_MISMATCH",
    "future_unavailable": "ABSTAIN_FUTURE_DATA",
    "entity_not_found": "ABSTAIN_ENTITY_NOT_FOUND",
    "not_released_as_of_cutoff": "ABSTAIN_NOT_RELEASED_AS_OF_CUTOFF",
}

def contains_any(text: str, tokens: List[str]) -> bool:
    text = text.lower()
    return any(str(t).lower() in text for t in tokens)


def diagnose_claim(case: Dict[str, Any], claim: Dict[str, Any], trace: Dict[str, Any]) -> Dict[str, Any]:
    availability = claim["availability"]
    source = claim.get("source", "")
    sources = executed_sources(trace)
    ans_text = answer_text(trace)
    found = evidence_found_for_claim(claim, trace)

    result = {
        "question_id": case["question_id"],
        "claim_id": claim["claim_id"],
        "claim": claim["label"],
        "availability": availability,
        "source": source,
        "source_executed": source in sources if source in {"rdb", "graph", "vector", "ontology"} else None,
        "evidence_found": found,
        "diagnosis": None,
        "severity": "INFO"
    }

    # 0) Query validity / expected abstain
    if availability in ABSTAIN_MAP:
        expected = ABSTAIN_MAP[availability].lower()
        tokens = claim.get("abstain_tokens", [expected])
        if expected in ans_text and contains_any(ans_text, tokens):
            result["diagnosis"] = "EXPECTED_ABSTAIN"
        else:
            result["diagnosis"] = "MISSED_ABSTAIN"
            result["severity"] = "CRITICAL"
        return result

    # 1) Deprecated definition
    if availability == "deprecated_definition":
        forbidden = [x.lower() for x in case.get("forbidden_fields", [])]
        required_def = case.get("required_definition_tokens", [])
        # 폐기 필드의 이름을 답변에서 "사용하지 않는다"고 설명하는 것은 정상이다.
        # 실제 오용 여부는 query plan / SQL / SPARQL / retrieval trace에서만 검사한다.
        execution_text = "\n".join([
            json.dumps(trace.get("plan", []), ensure_ascii=False, default=str),
            json.dumps(trace.get("step_results", {}), ensure_ascii=False, default=str),
            " ".join(trace.get("trace_messages", []))
        ]).lower()
        used_forbidden = contains_any(execution_text, forbidden) if forbidden else False
        definition_explained = contains_any(ans_text, required_def) if required_def else True

        if used_forbidden:
            result["diagnosis"] = "STALE_DEFINITION"
            result["severity"] = "CRITICAL"
        elif not definition_explained:
            result["diagnosis"] = "MISSING_DEFINITION_EXPLANATION"
            result["severity"] = "WARN"
        else:
            result["diagnosis"] = "PASS"
        return result

    # 2) Internal data should exist
    if availability == "available":
        if source in {"rdb", "graph", "vector"} and source not in sources:
            result["diagnosis"] = "ROUTING_MISS"
            result["severity"] = "ERROR"
            return result

        if not found:
            result["diagnosis"] = "RETRIEVAL_MISS"
            result["severity"] = "ERROR"
            return result

        answer_tokens = claim.get("answer_tokens", [])
        if answer_tokens and not contains_any(ans_text, answer_tokens):
            result["diagnosis"] = "GENERATION_OMISSION"
            result["severity"] = "ERROR"
            return result

        result["diagnosis"] = "PASS"
        return result

    # 3) External/partial data
    if availability in {"external_required", "partial", "not_available_by_design"}:
        gap_tokens = claim.get("gap_tokens", ["없", "부족", "외부", "필요", "미제공"])
        gap_acknowledged = contains_any(ans_text, gap_tokens)

        # 데이터가 없다고 golden이 말하는데 answer가 확정적으로 값을 만들면 위험
        if found and availability == "external_required":
            # found가 실제 외부 데이터 ingest 성공을 의미할 수도 있으므로,
            # 운영 환경에서는 provenance(source_type)를 같이 확인해야 한다.
            result["diagnosis"] = "EXTERNAL_EVIDENCE_PRESENT_OR_SCHEMA_CHECK_NEEDED"
            result["severity"] = "INFO"
        elif gap_acknowledged:
            result["diagnosis"] = "EXTERNAL_DATA_REQUIRED"
        else:
            result["diagnosis"] = "UNSUPPORTED_INFERENCE"
            result["severity"] = "CRITICAL"
        return result

    result["diagnosis"] = "UNCLASSIFIED"
    result["severity"] = "WARN"
    return result


## 5. 문항 단위 평가

점수는 보조 지표다. 실무적으로는 `failure_codes`와 Claim matrix가 더 중요하다.


In [ ]:

def evaluate_case(case: Dict[str, Any], trace: Dict[str, Any]) -> Dict[str, Any]:
    claim_results = [diagnose_claim(case, c, trace) for c in case["claims"]]

    failure_codes = sorted({
        r["diagnosis"] for r in claim_results
        if r["diagnosis"] not in {
            "PASS", "EXPECTED_ABSTAIN", "EXTERNAL_DATA_REQUIRED",
            "EXTERNAL_EVIDENCE_PRESENT_OR_SCHEMA_CHECK_NEEDED"
        }
    })

    critical = [r for r in claim_results if r["severity"] == "CRITICAL"]
    errors = [r for r in claim_results if r["severity"] == "ERROR"]

    expected_routes = set(case.get("expected_routes", []))
    observed_routes = executed_sources(trace)
    route_recall = (
        len(expected_routes & observed_routes) / len(expected_routes)
        if expected_routes else 1.0
    )

    positive = {"PASS", "EXPECTED_ABSTAIN", "EXTERNAL_DATA_REQUIRED",
                "EXTERNAL_EVIDENCE_PRESENT_OR_SCHEMA_CHECK_NEEDED"}
    claim_score = sum(r["diagnosis"] in positive for r in claim_results) / max(1, len(claim_results))

    policy_pass = len(critical) == 0
    overall_pass = policy_pass and len(errors) == 0

    return {
        "question_id": case["question_id"],
        "golden_data_status": case["golden_data_status"],
        "expected_action": case["expected_action"],
        "route_recall": round(route_recall, 3),
        "claim_score": round(claim_score, 3),
        "policy_pass": policy_pass,
        "overall_pass": overall_pass,
        "failure_codes": failure_codes,
        "claim_results": claim_results,
    }


def claim_matrix(results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for result in results:
        rows.extend(result["claim_results"])
    return pd.DataFrame(rows)


def summary_table(results: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "question_id": r["question_id"],
            "golden_status": r["golden_data_status"],
            "expected_action": r["expected_action"],
            "route_recall": r["route_recall"],
            "claim_score": r["claim_score"],
            "policy_pass": r["policy_pass"],
            "overall_pass": r["overall_pass"],
            "failure_codes": ", ".join(r["failure_codes"]) or "-"
        }
        for r in results
    ])


## 6. Synthetic Trace

실제 `app`을 연결하기 전 evaluator 자체가 원하는 진단을 내리는지 확인하기 위한 샘플이다.


In [ ]:

def synthetic_trace_for(case_id: str) -> Dict[str, Any]:
    base = {
        "nodes_executed": ["receive_question", "analyze_intent", "verify_intent", "plan_query"],
        "intent": {},
        "verified_intent": {},
        "plan": [],
        "route": {},
        "step_results": {},
        "merged_rows": [],
        "answer": {},
        "errors": [],
        "trace_messages": []
    }

    if case_id == "Q2":
        base["nodes_executed"] += ["rdb_search", "merge_results", "generate_answer"]
        row = {
            "pd_no": "KR103502GB66",
            "isu_dt": "20210610",
            "mat_dt": "20310610",
            "remaining_days": 1754,
            "srfc_irt": 2.0,
            "after_tax_yield": 3.9279,
            "info_base_dt": "20260821"
        }
        base["step_results"] = {"rdb_search": {"S1": {"rows": [row], "count": 1}}}
        base["merged_rows"] = [row]
        base["answer"] = {"answer": "KR103502GB66, 발행일 20210610, 만기일 20310610, 잔존일수 1754일, 표면금리 2%, 세후수익률 3.9279%, 기준일 20260821"}
        return base

    if case_id == "Q11":
        base["nodes_executed"] += ["rdb_search", "merge_results", "generate_answer"]
        row = {"mat_dt": "20300910", "crd_grd": "AAA", "applied_yield": 4.233}
        base["step_results"] = {"rdb_search": {"S1": {"rows": [row], "sql": "SELECT mat_dt, crd_grd, applied_yield FROM bond"}}}
        base["merged_rows"] = [row]
        base["answer"] = {"answer": "BUYABLE_QUANTITY는 사용하지 않고 만기 미도래 채권을 구매가능으로 재정의한다."}
        return base

    if case_id == "Q23":
        base["nodes_executed"] += ["rdb_search", "graph_search", "vector_search", "merge_results", "generate_answer"]
        base["step_results"] = {
            "rdb_search": {"S1": {"rows": [{"themes": ["우주항공", "방산"]}]}},
            "graph_search": {"S2": {"rows": []}},
            "vector_search": {"S3": {"rows": []}}
        }
        base["merged_rows"] = [{"themes": ["우주항공", "방산"]}]
        base["answer"] = {"answer": "현재 우주항공/방산 테마는 확인되지만 최근 6개월 이력은 시점·날짜 데이터가 없어 외부 이력이 필요하다. 현재 편입과 뉴스 언급을 구분하려면 편입내역이 필요하다."}
        return base

    if case_id == "Q24":
        base["nodes_executed"] += ["graph_search", "rdb_search", "vector_search", "merge_results", "generate_answer"]
        base["step_results"] = {
            "graph_search": {"S1": {"rows": [], "note": "SUBSIDIARY_OF / HOLDS_SECURITY unavailable"}},
            "rdb_search": {"S2": {"rows": [{"pd_net_tamt": 1234567890}]}},
            "vector_search": {"S3": {"rows": [], "note": "risk document unavailable"}}
        }
        base["merged_rows"] = [{"pd_net_tamt": 1234567890}]
        base["answer"] = {"answer": "AUM은 내부 데이터로 확인 가능하다. 자회사 관계는 DART 등 외부 데이터가 필요하고, ETF 편입내역·편입비중도 내부에 없어 외부 편입내역이 필요하다. 위험요인은 투자설명서 문서 확보가 필요하다."}
        return base

    if case_id == "Q31":
        base["nodes_executed"] += ["generate_answer"]
        base["answer"] = {"answer": "ABSTAIN_INVALID_TAXONOMY: AAAA는 존재하지 않는 신용등급 taxonomy다."}
        return base

    if case_id == "Q35":
        base["nodes_executed"] += ["generate_answer"]
        base["answer"] = {"answer": "ABSTAIN_DOMAIN_MISMATCH: VOO는 ETF이며 issuedBy의 대상 domain은 Bond이므로 회사채 발행 주체로 볼 수 없다."}
        return base

    return base


## 7. 샘플 실행


In [ ]:

sample_results = []
sample_traces = {}

for case in SAMPLE_GOLDEN:
    trace = synthetic_trace_for(case["question_id"])
    sample_traces[case["question_id"]] = trace
    sample_results.append(evaluate_case(case, trace))

summary_table(sample_results)


In [ ]:
claim_matrix(sample_results)[[
    "question_id", "claim_id", "claim", "availability",
    "source", "source_executed", "evidence_found", "diagnosis", "severity"
]]


## 8. 실제 LangGraph `app` 연결

프로젝트 환경에서 `app`을 import한 후 아래 셀을 사용한다.

예시:

```python
from agent.agent_core import app
```

그다음 `USE_SYNTHETIC = False`로 바꾼다.


In [ ]:
from agent.agent_core import app

In [ ]:

USE_SYNTHETIC = True  # 실제 app 연결 시 False

def run_golden_suite(cases, app=None):
    results = []
    traces = {}

    for case in cases:
        if USE_SYNTHETIC:
            trace = synthetic_trace_for(case["question_id"])
        else:
            if app is None:
                raise ValueError("USE_SYNTHETIC=False이면 app을 전달해야 한다.")
            trace = collect_agent_trace(app, case["question"])

        traces[case["question_id"]] = trace
        results.append(evaluate_case(case, trace))

    return results, traces


results, traces = run_golden_suite(SAMPLE_GOLDEN, app=globals().get("app"))
summary_table(results)


## 9. 실패 문항만 Root Cause 출력


In [ ]:

def root_cause_report(results):
    rows = []
    for r in results:
        for c in r["claim_results"]:
            if c["diagnosis"] not in {
                "PASS", "EXPECTED_ABSTAIN", "EXTERNAL_DATA_REQUIRED",
                "EXTERNAL_EVIDENCE_PRESENT_OR_SCHEMA_CHECK_NEEDED"
            }:
                rows.append({
                    "question_id": c["question_id"],
                    "claim_id": c["claim_id"],
                    "claim": c["claim"],
                    "golden_availability": c["availability"],
                    "expected_source": c["source"],
                    "source_executed": c["source_executed"],
                    "evidence_found": c["evidence_found"],
                    "root_cause": c["diagnosis"],
                    "severity": c["severity"],
                })
    return pd.DataFrame(rows)

root_cause_report(results)


## 10. 의도적으로 오류를 넣어 진단 확인

Q24에서 GraphDB를 아예 실행하지 않았다고 가정해 `ROUTING_MISS`가 잡히는지 확인한다.


In [ ]:

bad_q24 = synthetic_trace_for("Q24")
bad_q24["nodes_executed"] = [x for x in bad_q24["nodes_executed"] if x != "rdb_search"]
bad_q24["step_results"].pop("rdb_search", None)
bad_q24["merged_rows"] = []
bad_q24["answer"] = {
    "answer": "자회사와 ETF 편입관계는 외부 데이터가 필요하다. AUM은 확인하지 못했다."
}

q24_case = next(c for c in SAMPLE_GOLDEN if c["question_id"] == "Q24")
bad_result = evaluate_case(q24_case, bad_q24)

pd.DataFrame(bad_result["claim_results"])


## 11. 운영 단계 권장 확장

현재 샘플은 row key / answer token 기반의 deterministic evaluator다.

실제 운영에서는 아래 두 평가를 추가하는 것이 좋다.

### A. Deterministic Judge
- route
- SQL/SPARQL 실행 여부
- 컬럼 존재 여부
- row count
- 기준일
- taxonomy/domain
- abstain reason

### B. LLM-as-a-Judge
- 자연어 answer가 Claim을 실제로 충족했는지
- 근거와 Claim 사이 entailment가 성립하는지
- “상품명에 엔비디아 포함”을 “엔비디아 편입”으로 과잉 추론했는지
- 부분산출/외부데이터 필요를 적절하게 표현했는지

LLM Judge의 출력도 다음처럼 Claim 단위로 강제한다.

```json
{
  "claim_id": "C2",
  "verdict": "FAIL",
  "failure_code": "UNSUPPORTED_INFERENCE",
  "evidence_used": ["pd_abrv_nm"],
  "missing_evidence": ["holding_security", "holding_weight", "holding_as_of"],
  "reason": "상품명 일치는 실제 편입관계를 증명하지 않는다."
}
```

이렇게 하면 최종적으로 **데이터 엔지니어링 문제 / 검색 문제 / Agent 로직 문제 / 생성 문제**를 분리해서 고칠 수 있다.
